In [1]:
import sys as __harness_sys
from pathlib import Path as __harness_Path
__harness_sys.path.insert(0, 'C:\\Users\\jacob\\Documents\\AVANTECK.TEAM\\mx-m6hn-agentic_systems-feature-agentic-lab@f711b6b9bbe\\tutorials')
__harness_sys.path.insert(0, 'C:\\Users\\jacob\\Documents\\AVANTECK.TEAM\\mx-m6hn-agentic_systems-feature-agentic-lab@f711b6b9bbe\\scripts')
from _offline import enforce_offline as __guard, blocked_attempts as __blocked
__guard()
import capture_runresults_213 as __capture
__capture.install()
import agentic_systems as __harness_toolkit
assert __harness_toolkit.__version__ == "2.1.3"
assert "site-packages" in __harness_Path(__harness_toolkit.__file__).parts

# Environment: episodios y pasos

Dos inputs → la misma unidad por paso → dos outputs e historia de episodio.

API **2.1.3**. Cada ejemplo muestra una entrada y su resultado. Python Runtime realiza trabajo determinista.

In [2]:
import agentic_systems as toolkit


In [3]:
@toolkit.tool
def increment(value: int) -> dict:
    """Suma uno de forma determinista."""
    return {"value": value + 1}

@toolkit.tool
def double(value: int) -> dict:
    """Duplica el valor."""
    return {"value": value * 2}

arithmetic = toolkit.skill(name="arithmetic", tools=[increment, double])
increase = toolkit.Action("increase", "increment")
multiply = toolkit.Action("multiply", "double")

In [4]:
agent = toolkit.Agent("calculator", toolkit.Sequence((increase, multiply)), skills=(arithmetic,))
application = toolkit.System("team", toolkit.Sequence((agent,)))
episode = toolkit.Environment(name="two_steps", unit=application, records=({"value": 3}, {"value": -3}))
result = episode.run()
assert result.ok and result.data["outputs"] == [{"value": 8}, {"value": -4}]
assert result.data["summary"]["memory"] == {"processed": 2}
print({"input": [3, -3], "process": "un paso por registro", "output": result.data["outputs"]})

{'input': [3, -3], 'process': 'un paso por registro', 'output': [{'value': 8}, {'value': -4}]}


Input: importes 2, 3, 100. Proceso: observe combina registro y memoria; update conserva el total; stop termina al llegar a 5. Output: 2 y 5, sin procesar 100. La declaración se comparte con examples/episode_memory.

In [5]:
import agentic_systems as toolkit

def build_episode(backend='native'):
    worker = toolkit.Agent('sum', toolkit.Action('sum', lambda d: {'total': d['previous'] + d['amount']}), backend=backend)
    return toolkit.Environment('ledger', worker, ({'amount': 2}, {'amount': 3}, {'amount': 100}), memory={'total': 0}, observe=lambda row, memory: {'amount': row['amount'], 'previous': memory['total']}, update=lambda memory, row, result: result.data, stop=lambda memory, result: memory['total'] >= 5)
import inspect
print(inspect.getsource(build_episode))
episode = build_episode()
prepared = episode.compile()
result = prepared.run()
assert result.ok and result.data['summary']['memory'] == {'total':5}
assert result.data['summary']['terminated']
assert (await episode.arun()).data['outputs'] == result.data['outputs']
assert (await prepared.arun()).data['outputs'] == result.data['outputs']
print({'input':list(episode.records),'process':prepared.inspect(),'output':result.data['outputs']})

def build_episode(backend='native'):
    worker = toolkit.Agent('sum', toolkit.Action('sum', lambda d: {'total': d['previous'] + d['amount']}), backend=backend)
    return toolkit.Environment('ledger', worker, ({'amount': 2}, {'amount': 3}, {'amount': 100}), memory={'total': 0}, observe=lambda row, memory: {'amount': row['amount'], 'previous': memory['total']}, update=lambda memory, row, result: result.data, stop=lambda memory, result: memory['total'] >= 5)

{'input': [{'amount': 2}, {'amount': 3}, {'amount': 100}], 'process': {'name': 'ledger', 'kind': 'environment', 'records': 3, 'max_steps': None, 'memory': True, 'observe': True, 'update': True, 'stop': True, 'unit': {'name': 'sum', 'kind': 'agent', 'backend': 'native', 'native_type': None, 'children': [{'name': 'sum', 'kind': 'action', 'backend': 'python', 'native_type': None, 'children': []}]}}, 'output': [{'total': 2}, {'total': 5}]}


Cada run abre un episodio nuevo. Sin observe, cada registro es el input. observe y update conectan memoria real entre pasos; stop permite terminar el episodio. Las transiciones están en `result.data['history']` y las ejecuciones de las unidades en `result.children`.

In [6]:
__capture.save('C:\\Users\\jacob\\Documents\\AVANTECK.TEAM\\mx-m6hn-agentic_systems-feature-agentic-lab@f711b6b9bbe\\revisions\\2.1.3\\learning-evidence-elevated\\notebook-runresults\\20261023T044738543256\\08-04_environment.json', 'tutorials/core/04_environment.ipynb')
assert __blocked() == 0